# Swedish ISK vs a taxable investment account

This notebook compares the **same SWDA investment** in a Swedish ISK and a regular taxable account (depå/AF). It uses the SEK-converted price history already prepared in [`../world_fund_comparison.ipynb`](../world_fund_comparison.ipynb), then lets you explore historical holding periods and forward-looking return scenarios.

**Start here:** run the notebook from top to bottom. The historical section shows how often ISK finished ahead over past SWDA periods. The scenario section gives a heatmap and interactive controls for different starting amounts, monthly savings, expected returns, dividend yields, and horizons.

The result is a tax comparison, not a forecast. It assumes the investor is an individual tax resident in Sweden and the simulated account gets the full ISK/KF/PEPP tax-free allowance unless you reduce it below. Rates and tax law can change.

## Data and what is being compared

The existing notebook labels the Xetra listing **EUNL**. It is the same iShares Core MSCI World share class as **SWDA** (ISIN IE00B4L5Y983), whose income is accumulating. The input series is `ishares_close_sek`: EUNL's EUR close converted into SEK with the ECB reference rate. The source file is `../data/aligned_performance_sek.csv`.

The rolling-history section uses completed calendar years from 2012 through 2025, since ISK began in 2012. The accumulating SWDA share class does not pay a cash dividend to its holder, so its default Swedish investor-level dividend yield is **0%**. You can enter a yield to model a distributing investment. The fund's internal withholding taxes and ongoing costs are already reflected in its history; they are not subtracted again.

Price history is a sample, not a forecast. One daily/monthly price path cannot show every possible market sequence, and historical returns may not repeat.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML

plt.style.use("seaborn-v0_8-whitegrid")

# Main inputs you can edit
# The notebook may be opened from either this subfolder or the project root.
DATA_CANDIDATES = [base / "data" / "aligned_performance_sek.csv"
                   for base in (Path.cwd(), *Path.cwd().parents)]
DATA_FILE = next((path for path in DATA_CANDIDATES if path.is_file()), DATA_CANDIDATES[0])
CAPITAL_TAX_RATE = 0.30
DIVIDEND_TAX_RATE = 0.30
CAPITAL_LOSS_TAX_RELIEF_RATE = 0.21  # simplified: 70% loss deduction x 30% tax value
FUND_IMPUTED_INCOME_RATE = 0.004  # taxable-account fund units: 0.4% of value at 1 Jan
FUND_TAX_RATE = FUND_IMPUTED_INCOME_RATE * CAPITAL_TAX_RATE  # 0.12% effective annual tax

# Current law for 2026. Future statutory rates are unknown, so projections hold these constant.
FORECAST_ISK_SCHABLON_RATE = 0.0355  # 2026 schablonintäkt rate, before the 30% capital tax
FORECAST_ISK_ALLOWANCE_SEK = 300_000.0
ISK_ALLOWANCE_SHARE_AVAILABLE = 1.0  # set to 0 if other ISK/KF/PEPP savings already use the allowance

# Historical rolling-window settings
HISTORICAL_START_AMOUNTS_SEK = [100_000, 300_000, 600_000, 1_000_000]
HISTORICAL_MONTHLY_CONTRIBUTION_SEK = 0.0
HISTORICAL_DISTRIBUTION_YIELD = 0.0  # SWDA is accumulating; set a yield to model distributions
HISTORICAL_HORIZONS_YEARS = [1, 3, 5, 10, 12]

# Forward scenario grid settings
GRID_INITIAL_SEK = 300_000.0
GRID_MONTHLY_SEK = 5_000.0
GRID_DIVIDEND_YIELD = 0.0
GRID_RETURNS = [-0.10, -0.05, 0.00, 0.03, 0.05, 0.07, 0.10, 0.12, 0.15]
GRID_HORIZONS_YEARS = [1, 3, 5, 10, 15, 20, 30]

if not DATA_FILE.exists():
    searched = "\n".join(str(path) for path in DATA_CANDIDATES)
    raise FileNotFoundError(f"Could not find aligned_performance_sek.csv. Searched:\n{searched}")

FileNotFoundError: Could not find /Users/davidezamboni/Documents/Swedish-Brokers-Comparison/isk_vs_taxable_account/data/aligned_performance_sek.csv. Run this notebook from the project folder.

## Tax rules used

- **ISK:** each year's capital base is the average of the values at the start of the four quarters plus deposits made during that year, divided by four. A statutory schablonintäkt rate is applied; 30% of that deemed income is tax. The simulation subtracts the available tax-free allowance from the capital base. For 2025 the allowance was SEK 150,000; for 2026 it is SEK 300,000 across ISK, capital insurance, and PEPP combined. The historical backtest uses the statutory ISK rates for each year. Forward scenarios assume the 2026 rate (3.55%) and allowance (SEK 300,000) continue unchanged.
- **Regular account:** positive realized gains are taxed at 30% when the investment is sold. For market-listed fund units, the model compares the actual average acquisition cost with the Swedish schablon method (cost basis = 20% of sale proceeds) and uses the lower tax. A realized loss is assigned a simplified 21% tax value (70% deductible × 30%); actual relief depends on your other capital income and deductions. Fund units also have a separate annual tax: 0.4% of their value on 1 January is taxed at 30%, an effective 0.12%. Cash dividends are taxed at 30%; the remaining dividend is treated as reinvested. The default yield is zero for accumulating SWDA.
- Annual taxes are modeled as withdrawals from the invested portfolio at each year-end, so they no longer earn later returns. For annual tax expenses in the regular account, the model also allows for the small realized-gain tax when shares are sold to fund that expense.

For simplicity, the model assumes the full legal ISK allowance is available to the simulated account (change `ISK_ALLOWANCE_SHARE_AVAILABLE` if not), ignores transaction fees and approximates capital-loss relief at 21% (your actual relief may differ). It treats dividend yield as an editable annual estimate and uses the same underlying total-return path for both accounts. This is an educational comparison, not a personal tax calculation.

In [ ]:
# Historical effective schablonintäkt rates by tax year. These are the rates applied
# to the ISK capital base; the actual tax is 30% of the deemed income.
ISK_SCHABLON_RATE_BY_YEAR = {
    2012: 0.0165, 2013: 0.0149, 2014: 0.0209, 2015: 0.0090,
    2016: 0.0140, 2017: 0.0125, 2018: 0.0149, 2019: 0.0151,
    2020: 0.0125, 2021: 0.0125, 2022: 0.0125, 2023: 0.0294,
    2024: 0.0362, 2025: 0.0296, 2026: 0.0355,
}
ISK_ALLOWANCE_BY_YEAR = {2025: 150_000.0, 2026: 300_000.0}

source = pd.read_csv(DATA_FILE, parse_dates=["date"])
if "ishares_close_sek" not in source.columns:
    raise ValueError("The source CSV does not contain the expected ishares_close_sek column.")
source["ishares_close_sek"] = pd.to_numeric(source["ishares_close_sek"], errors="coerce")
source = source.dropna(subset=["date", "ishares_close_sek"]).sort_values("date")
if source["date"].duplicated().any() or (source["ishares_close_sek"] <= 0).any():
    raise ValueError("The source history contains duplicate dates or non-positive prices.")

all_monthly_prices = (source.set_index("date")["ishares_close_sek"]
                      .resample("ME").last().dropna().sort_index())
latest_source_date = source["date"].max()
historical_prices = all_monthly_prices.loc["2012-01-01":"2025-12-31"]
if historical_prices.empty:
    raise ValueError("No SWDA/EUNL SEK observations were found from 2012 onward.")

print(f"Source data: {source['date'].min().date()} to {latest_source_date.date()} ({len(source):,} aligned weekdays)")
print(f"Completed-year backtest: {historical_prices.index.min().date()} to {historical_prices.index.max().date()}")
display(pd.DataFrame({
    "Input": ["SWDA/EUNL price in SEK", "Latest source observation", "Taxable-account fund tax"],
    "Value": ["ishares_close_sek", latest_source_date.date(), f"{FUND_TAX_RATE:.2%} of value at 1 January per year"],
}))

## Simulation engine

The historical calculations buy at the first month-end and add the optional monthly contribution at later month-ends. At each year-end, the model calculates that year's ISK capital base from the four quarter-opening values and deposits, then applies the relevant rules for that tax year. The regular account pays its annual fund/dividend taxes, and both accounts are valued after tax at the selected horizon. The regular account then pays 30% on any remaining positive capital gain when sold.

Quarter-opening values use the last monthly observation before the quarter begins. The source series was converted using ECB reference rates and prices carried forward across non-trading days, as described in the original comparison notebook.

In [ ]:
def isk_parameters(tax_year, forward_assumptions=False):
    # Return that tax year's schablonintäkt rate and available allowance.
    if tax_year < 2012:
        return 0.0, 0.0
    if tax_year in ISK_SCHABLON_RATE_BY_YEAR:
        rate = ISK_SCHABLON_RATE_BY_YEAR[tax_year]
    elif forward_assumptions and tax_year > 2026:
        rate = FORECAST_ISK_SCHABLON_RATE
    else:
        raise ValueError(f"No ISK rate configured for {tax_year}.")

    if tax_year in ISK_ALLOWANCE_BY_YEAR:
        legal_allowance = ISK_ALLOWANCE_BY_YEAR[tax_year]
    elif forward_assumptions and tax_year > 2026:
        legal_allowance = FORECAST_ISK_ALLOWANCE_SEK
    else:
        legal_allowance = 0.0
    return rate, legal_allowance * ISK_ALLOWANCE_SHARE_AVAILABLE


def quarter_opening_value(series, monthly_values, start_date, end_date, tax_year, month, account="isk"):
    quarter_start = pd.Timestamp(year=tax_year, month=month, day=1)
    if quarter_start > pd.Timestamp(end_date):
        return 0.0
    prior_dates = series.index[series.index < quarter_start]
    if len(prior_dates) == 0:
        return 0.0
    prior_date = prior_dates[-1]
    if prior_date < start_date:
        return 0.0
    value = monthly_values.get(prior_date, {})
    return float(value.get(account, 0.0))


def pay_taxable_account_tax(value, basis, tax_due):
    # Sell enough units to pay the annual bill, allowing for gain tax on those units.
    value = max(0.0, float(value))
    basis = max(0.0, float(basis))
    tax_due = max(0.0, float(tax_due))
    if tax_due == 0 or value == 0:
        return value, basis, 0.0, tax_due

    actual_gain_per_sek_sold = (value - basis) / value
    if actual_gain_per_sek_sold >= 0:
        actual_tax_per_sek_sold = actual_gain_per_sek_sold * CAPITAL_TAX_RATE
    else:
        actual_tax_per_sek_sold = actual_gain_per_sek_sold * CAPITAL_LOSS_TAX_RELIEF_RATE
    schablon_tax_per_sek_sold = 0.80 * CAPITAL_TAX_RATE
    tax_per_sek_sold = min(actual_tax_per_sek_sold, schablon_tax_per_sek_sold)
    sale_needed = tax_due / (1.0 - tax_per_sek_sold)
    sale = min(value, sale_needed)
    basis_removed = basis * sale / value
    realized_tax_or_relief = sale * tax_per_sek_sold
    amount_available_for_bill = sale - realized_tax_or_relief
    unpaid = max(0.0, tax_due - amount_available_for_bill)
    return value - sale, basis - basis_removed, realized_tax_or_relief, unpaid


def simulate_accounts(prices, start_date, end_date, initial_sek=300_000.0,
                      monthly_sek=0.0, dividend_yield=0.0,
                      forward_assumptions=False):
    # Compare ISK and regular-account balances over one price path.
    start_date = pd.Timestamp(start_date)
    end_date = pd.Timestamp(end_date)
    path = prices.loc[(prices.index >= start_date) & (prices.index <= end_date)].dropna()
    if path.empty:
        raise ValueError("The selected dates do not overlap the price series.")
    if initial_sek < 0 or monthly_sek < 0 or dividend_yield < 0:
        raise ValueError("Investment amounts and dividend yield must be non-negative.")

    start_date = path.index[0]
    isk_units = float(initial_sek) / float(path.iloc[0])
    regular_units = float(initial_sek) / float(path.iloc[0])
    regular_basis = float(initial_sek)
    deposits_by_year = {start_date.year: float(initial_sek)}
    monthly_values = {}
    month_values_by_year = {}
    isk_tax_total = 0.0
    isk_unpaid_tax = 0.0
    regular_fund_tax_total = 0.0
    regular_dividend_tax_total = 0.0
    regular_sale_tax_total = 0.0
    regular_unpaid_tax = 0.0
    annual_rows = []

    for i, (date, price) in enumerate(path.items()):
        price = float(price)
        if i > 0 and monthly_sek:
            isk_units += float(monthly_sek) / price
            regular_units += float(monthly_sek) / price
            regular_basis += float(monthly_sek)
            deposits_by_year[date.year] = deposits_by_year.get(date.year, 0.0) + float(monthly_sek)

        isk_value = isk_units * price
        regular_value = regular_units * price
        monthly_values[date] = {"isk": isk_value, "regular": regular_value}
        month_values_by_year.setdefault(date.year, []).append(regular_value)

        is_year_end = (i == len(path) - 1) or (path.index[i + 1].year != date.year)
        if not is_year_end:
            continue

        tax_year = int(date.year)
        q_values = [quarter_opening_value(prices, monthly_values, start_date, path.index[-1], tax_year, month)
                    for month in (1, 4, 7, 10)]
        deposits = float(deposits_by_year.get(tax_year, 0.0))
        capital_base = (sum(q_values) + deposits) / 4.0
        schablon_rate, allowance = isk_parameters(tax_year, forward_assumptions)
        isk_tax = max(0.0, capital_base - allowance) * schablon_rate * CAPITAL_TAX_RATE
        value_now = isk_units * price
        paid_from_isk = min(value_now, isk_tax)
        isk_units = max(0.0, value_now - paid_from_isk) / price
        isk_unpaid_tax += max(0.0, isk_tax - paid_from_isk)
        isk_tax_total += isk_tax

        regular_value = regular_units * price
        jan_opening_value = quarter_opening_value(prices, monthly_values, start_date, path.index[-1], tax_year, 1, account="regular")
        fund_tax = jan_opening_value * FUND_TAX_RATE
        gross_dividend = (sum(month_values_by_year.get(tax_year, [])) / 12.0) * float(dividend_yield)
        dividend_tax = gross_dividend * DIVIDEND_TAX_RATE
        regular_basis += gross_dividend * (1.0 - DIVIDEND_TAX_RATE)
        annual_tax_due = fund_tax + dividend_tax
        regular_value_after, regular_basis, sale_tax, unpaid = pay_taxable_account_tax(
            regular_value, regular_basis, annual_tax_due
        )
        regular_units = regular_value_after / price
        regular_sale_tax_total += sale_tax
        regular_unpaid_tax += unpaid
        regular_fund_tax_total += fund_tax
        regular_dividend_tax_total += dividend_tax

        # The annual tax payment is treated as a year-end portfolio withdrawal.
        monthly_values[date] = {
            "isk": isk_units * price,
            "regular": regular_units * price,
        }
        annual_rows.append({
            "tax_year": tax_year,
            "quarter_open_values_sek": q_values,
            "deposits_sek": deposits,
            "isk_capital_base_sek": capital_base,
            "isk_schablon_rate": schablon_rate,
            "isk_allowance_sek": allowance,
            "isk_tax_sek": isk_tax,
            "regular_fund_tax_sek": fund_tax,
            "regular_gross_dividend_sek": gross_dividend,
            "regular_dividend_tax_sek": dividend_tax,
            "regular_realized_gain_tax_to_fund_annual_tax_sek": sale_tax,
        })

    last_price = float(path.iloc[-1])
    isk_gross_end = isk_units * last_price
    regular_gross_end = regular_units * last_price
    actual_gain = regular_gross_end - regular_basis
    actual_basis_tax = (actual_gain * CAPITAL_TAX_RATE if actual_gain >= 0
                        else actual_gain * CAPITAL_LOSS_TAX_RELIEF_RATE)
    schablon_basis_tax = regular_gross_end * 0.80 * CAPITAL_TAX_RATE
    final_capital_gains_tax = min(actual_basis_tax, schablon_basis_tax)
    final_capital_gain = max(0.0, actual_gain)
    regular_after_tax_end = regular_gross_end - final_capital_gains_tax - regular_unpaid_tax
    isk_after_tax_end = isk_gross_end - isk_unpaid_tax
    return {
        "start_date": path.index[0],
        "end_date": path.index[-1],
        "years": (path.index[-1] - path.index[0]).days / 365.25,
        "contributions_sek": sum(deposits_by_year.values()),
        "isk_gross_end_sek": isk_gross_end,
        "isk_tax_sek": isk_tax_total,
        "isk_unpaid_tax_sek": isk_unpaid_tax,
        "isk_after_tax_end_sek": isk_after_tax_end,
        "regular_gross_end_sek": regular_gross_end,
        "regular_fund_tax_sek": regular_fund_tax_total,
        "regular_dividend_tax_sek": regular_dividend_tax_total,
        "regular_annual_tax_sale_gain_tax_sek": regular_sale_tax_total,
        "regular_final_capital_gain_sek": final_capital_gain,
        "regular_final_capital_gains_tax_sek": final_capital_gains_tax,
        "regular_unpaid_tax_sek": regular_unpaid_tax,
        "regular_after_tax_end_sek": regular_after_tax_end,
        "isk_minus_regular_sek": isk_after_tax_end - regular_after_tax_end,
        "annual_detail": pd.DataFrame(annual_rows),
    }

print("Tax schedule and account simulation functions are ready.")

## Historical SWDA rolling windows

Each window invests the same starting amount in both accounts, then holds the same SWDA SEK price path. The table reports the share of available start dates where ISK ended with more after-tax wealth, plus the median SEK difference. Because the tax-free allowance changed over time, each historical tax year uses its own statutory rules. These are overlapping start dates from one unusually strong equity-history sample, not independent trials or a forecast. The baseline models a new investment; it does not charge capital-gains tax for moving an already appreciated depå holding into ISK.

Edit the starting balances, monthly amount, dividend yield, and horizons in the inputs cell above, then rerun this section.

In [ ]:
historical_start_dates = historical_prices.index[historical_prices.index >= pd.Timestamp("2012-01-01")]
historical_records = []

for initial_amount in HISTORICAL_START_AMOUNTS_SEK:
    for horizon in HISTORICAL_HORIZONS_YEARS:
        for start in historical_start_dates:
            target = start + pd.DateOffset(years=int(horizon))
            eligible_ends = historical_prices.index[historical_prices.index <= target]
            if len(eligible_ends) == 0:
                continue
            end = eligible_ends[-1]
            if end <= start or (end - start).days < 300 * horizon:
                continue
            result = simulate_accounts(
                historical_prices, start, end,
                initial_sek=initial_amount,
                monthly_sek=HISTORICAL_MONTHLY_CONTRIBUTION_SEK,
                dividend_yield=HISTORICAL_DISTRIBUTION_YIELD,
                forward_assumptions=False,
            )
            historical_records.append({
                "initial_sek": initial_amount,
                "horizon_years": int(horizon),
                "start_date": result["start_date"],
                "end_date": result["end_date"],
                "actual_years": result["years"],
                "isk_after_tax_sek": result["isk_after_tax_end_sek"],
                "regular_after_tax_sek": result["regular_after_tax_end_sek"],
                "isk_minus_regular_sek": result["isk_minus_regular_sek"],
                "isk_tax_sek": result["isk_tax_sek"],
                "regular_fund_tax_sek": result["regular_fund_tax_sek"],
                "regular_dividend_tax_sek": result["regular_dividend_tax_sek"],
                "regular_final_capital_gains_tax_sek": result["regular_final_capital_gains_tax_sek"],
            })

historical_results = pd.DataFrame(historical_records)
if historical_results.empty:
    raise ValueError("No historical windows fit the chosen start dates and horizons.")
historical_summary = (historical_results
    .groupby(["initial_sek", "horizon_years"], as_index=False)
    .agg(windows=("isk_minus_regular_sek", "size"),
         isk_win_rate=("isk_minus_regular_sek", lambda x: (x > 0).mean()),
         median_difference_sek=("isk_minus_regular_sek", "median"),
         median_isk_end_sek=("isk_after_tax_sek", "median"),
         median_regular_end_sek=("regular_after_tax_sek", "median")))
historical_summary["isk_win_rate_pct"] = 100 * historical_summary["isk_win_rate"]

display(historical_summary[[
    "initial_sek", "horizon_years", "windows", "isk_win_rate_pct",
    "median_difference_sek", "median_isk_end_sek", "median_regular_end_sek"
]].round(1))

In [ ]:
# Visual comparison for the starting balance nearest SEK 300,000
plot_amount = min(HISTORICAL_START_AMOUNTS_SEK, key=lambda x: abs(x - 300_000))
plot_data = historical_summary[historical_summary.initial_sek == plot_amount].sort_values("horizon_years")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
axes[0].bar(plot_data.horizon_years.astype(str), plot_data.isk_win_rate_pct, color="#3b82f6")
axes[0].axhline(50, color="#555", linewidth=1, linestyle="--")
axes[0].set(title=f"How often ISK finished ahead (start SEK {plot_amount:,.0f})",
            xlabel="Holding period (years)", ylabel="Historical windows won by ISK (%)", ylim=(0, 100))

colors = ["#15803d" if x >= 0 else "#b91c1c" for x in plot_data.median_difference_sek]
axes[1].bar(plot_data.horizon_years.astype(str), plot_data.median_difference_sek, color=colors)
axes[1].axhline(0, color="#555", linewidth=1)
axes[1].set(title="Median after-tax wealth difference",
            xlabel="Holding period (years)", ylabel="ISK minus regular account (SEK)")
plt.tight_layout()
plt.show()

## Forward scenarios: when does ISK win?

The grid below uses constant **annual total returns in SEK**. A positive number means ISK has the higher after-tax ending value; a negative number means the regular account does. It assumes the 2026 ISK schablonintäkt rate and SEK 300,000 allowance continue for the full projection. The return and tax-rate settings are assumptions, not forecasts.

Change `GRID_INITIAL_SEK`, `GRID_MONTHLY_SEK`, `GRID_DIVIDEND_YIELD`, the return list, or horizon list in the inputs cell and rerun this section. The optional controls below let you inspect one scenario at a time.

In [ ]:
def constant_return_path(annual_return, years, start="2026-01-31"):
    periods = int(years) * 12
    dates = pd.date_range(start=pd.Timestamp(start), periods=periods + 1, freq="ME")
    elapsed_years = (dates - dates[0]).days / 365.25
    if annual_return <= -1:
        raise ValueError("Annual return must be greater than -100%.")
    values = 100.0 * (1.0 + float(annual_return)) ** elapsed_years
    return pd.Series(values, index=dates, name="price_sek")


def run_constant_scenario(annual_return, years, initial_sek, monthly_sek, dividend_yield):
    path = constant_return_path(annual_return, years)
    return simulate_accounts(
        path, path.index[0], path.index[-1],
        initial_sek=initial_sek, monthly_sek=monthly_sek,
        dividend_yield=dividend_yield, forward_assumptions=True,
    )

scenario_records = []
for annual_return in GRID_RETURNS:
    for horizon in GRID_HORIZONS_YEARS:
        result = run_constant_scenario(
            annual_return, horizon, GRID_INITIAL_SEK, GRID_MONTHLY_SEK, GRID_DIVIDEND_YIELD
        )
        scenario_records.append({
            "annual_return": annual_return,
            "horizon_years": horizon,
            "isk_after_tax_sek": result["isk_after_tax_end_sek"],
            "regular_after_tax_sek": result["regular_after_tax_end_sek"],
            "isk_minus_regular_sek": result["isk_minus_regular_sek"],
            "total_contributions_sek": result["contributions_sek"],
            "isk_tax_sek": result["isk_tax_sek"],
            "regular_fund_tax_sek": result["regular_fund_tax_sek"],
            "regular_dividend_tax_sek": result["regular_dividend_tax_sek"],
            "regular_final_capital_gains_tax_sek": result["regular_final_capital_gains_tax_sek"],
        })
scenario_results = pd.DataFrame(scenario_records)

fig, ax = plt.subplots(figsize=(12, 6))
gap_matrix = (scenario_results.pivot(index="horizon_years", columns="annual_return", values="isk_minus_regular_sek")
              .reindex(index=GRID_HORIZONS_YEARS, columns=GRID_RETURNS).to_numpy())
limit = max(1.0, float(np.nanmax(np.abs(gap_matrix))))
image = ax.imshow(gap_matrix, cmap="RdYlGn", aspect="auto", vmin=-limit, vmax=limit)
ax.set_xticks(range(len(GRID_RETURNS)), [f"{x:.0%}" for x in GRID_RETURNS])
ax.set_yticks(range(len(GRID_HORIZONS_YEARS)), [str(x) for x in GRID_HORIZONS_YEARS])
ax.set_xlabel("Assumed annual total return (SEK)")
ax.set_ylabel("Holding period (years)")
ax.set_title(f"After-tax ending wealth: ISK minus regular account\n"
             f"Initial SEK {GRID_INITIAL_SEK:,.0f} + SEK {GRID_MONTHLY_SEK:,.0f}/month; "
             f"annual distribution yield {GRID_DIVIDEND_YIELD:.1%}")
for i in range(len(GRID_HORIZONS_YEARS)):
    for j in range(len(GRID_RETURNS)):
        ax.text(j, i, f"{gap_matrix[i, j]:,.0f}", ha="center", va="center", fontsize=8,
                color="black" if abs(gap_matrix[i, j]) < 0.72 * limit else "white")
fig.colorbar(image, ax=ax, label="SEK difference (green = ISK higher)")
plt.tight_layout()
plt.show()

# Approximate the return where the account difference changes sign for each horizon.
break_even_rows = []
for horizon in GRID_HORIZONS_YEARS:
    subset = scenario_results[scenario_results.horizon_years == horizon].sort_values("annual_return")
    xs = subset.annual_return.to_numpy()
    ys = subset.isk_minus_regular_sek.to_numpy()
    crossing = None
    for i in range(len(xs) - 1):
        if ys[i] == 0:
            crossing = xs[i]
            break
        if ys[i] * ys[i + 1] < 0:
            crossing = xs[i] + (0 - ys[i]) * (xs[i + 1] - xs[i]) / (ys[i + 1] - ys[i])
            break
    break_even_rows.append({"horizon_years": horizon,
                            "approx_break_even_return": crossing,
                            "grid_min_difference_sek": float(ys.min()),
                            "grid_max_difference_sek": float(ys.max())})
break_even_table = pd.DataFrame(break_even_rows)
break_even_table["approx_break_even_return_pct"] = pd.to_numeric(
    break_even_table["approx_break_even_return"], errors="coerce"
) * 100
display(break_even_table[["horizon_years", "approx_break_even_return_pct",
                          "grid_min_difference_sek", "grid_max_difference_sek"]].round(1))

## Inspect one scenario interactively

If `ipywidgets` is installed and supported by your Jupyter front end, use the sliders to change the investment amount, monthly contribution, expected annual return, dividend yield, and horizon. If widgets are unavailable, the default example below still runs; edit the `DEFAULT_SCENARIO` values and rerun this cell.

In [ ]:
DEFAULT_SCENARIO = {
    "initial_sek": 300_000.0,
    "monthly_sek": 5_000.0,
    "annual_return": 0.07,
    "dividend_yield": 0.00,
    "years": 10,
}


def show_scenario(initial_sek, monthly_sek, annual_return, dividend_yield, years):
    result = run_constant_scenario(
        annual_return, int(years), float(initial_sek), float(monthly_sek), float(dividend_yield)
    )
    total_invested = result["contributions_sek"]
    comparison = pd.DataFrame([
        {"Account": "ISK", "After-tax ending value": result["isk_after_tax_end_sek"],
         "Taxes modeled": result["isk_tax_sek"]},
        {"Account": "Regular account", "After-tax ending value": result["regular_after_tax_end_sek"],
         "Taxes modeled": (result["regular_fund_tax_sek"] + result["regular_dividend_tax_sek"] +
                          result["regular_annual_tax_sale_gain_tax_sek"] +
                          result["regular_final_capital_gains_tax_sek"])},
    ])
    difference = result["isk_minus_regular_sek"]
    winner = "ISK" if difference > 0 else "Regular account"
    if np.isclose(difference, 0, atol=0.5):
        winner = "About even"
    summary = (f"{winner} ends higher by SEK {abs(difference):,.0f}. "
               f"Total contributions: SEK {total_invested:,.0f}. "
               f"Assumed return: {annual_return:.1%}/year; holding period: {int(years)} years.")
    display(HTML(f"<h4>{summary}</h4>"))
    display(comparison.style.format({"After-tax ending value": "SEK {:,.0f}", "Taxes modeled": "SEK {:,.0f}"}))
    tax_detail = result["annual_detail"].copy()
    if not tax_detail.empty:
        display(HTML("<b>Annual ISK and regular-account tax detail</b>"))
        display(tax_detail[["tax_year", "isk_capital_base_sek", "isk_schablon_rate",
                            "isk_allowance_sek", "isk_tax_sek", "regular_fund_tax_sek",
                            "regular_dividend_tax_sek"]].style.format({
            "isk_capital_base_sek": "SEK {:,.0f}", "isk_schablon_rate": "{:.2%}",
            "isk_allowance_sek": "SEK {:,.0f}", "isk_tax_sek": "SEK {:,.0f}",
            "regular_fund_tax_sek": "SEK {:,.0f}", "regular_dividend_tax_sek": "SEK {:,.0f}"
        }))

try:
    import ipywidgets as widgets

    controls = widgets.interactive(
        show_scenario,
        initial_sek=widgets.IntSlider(value=int(DEFAULT_SCENARIO["initial_sek"]), min=0, max=1_000_000,
                                      step=25_000, description="Initial SEK", continuous_update=False),
        monthly_sek=widgets.IntSlider(value=int(DEFAULT_SCENARIO["monthly_sek"]), min=0, max=20_000,
                                      step=1_000, description="Monthly SEK", continuous_update=False),
        annual_return=widgets.FloatSlider(value=DEFAULT_SCENARIO["annual_return"], min=-0.15, max=0.20,
                                          step=0.005, readout_format=".1%", description="Return",
                                          continuous_update=False),
        dividend_yield=widgets.FloatSlider(value=DEFAULT_SCENARIO["dividend_yield"], min=0.0, max=0.08,
                                           step=0.005, readout_format=".1%", description="Dividend yield",
                                           continuous_update=False),
        years=widgets.IntSlider(value=DEFAULT_SCENARIO["years"], min=1, max=30,
                                step=1, description="Years", continuous_update=False),
    )
    display(controls)
except ImportError:
    print("ipywidgets is unavailable here; showing the editable default scenario.")
    show_scenario(**DEFAULT_SCENARIO)

## What Swedish Reddit discussions tend to emphasize

These threads show recurring arguments, not representative polling or tax advice:

- **ISK as the easy default:** commenters often favor it for long-term equity funds because gains/rebalancing are not individually taxed and reporting is simpler. Some replies also emphasize that the annual tax is charged even in down years. [ISK or regular savings account (2024)](https://www.reddit.com/r/PrivatEkonomi/comments/1b8rhdm/)
- **Depå can win under low returns or delayed selling:** commenters point to tax deferral, low annual fund tax, and being able to choose when to realize gains. Some compare lifetime withdrawal plans rather than selling everything at once. [Depå or ISK (2024)](https://www.reddit.com/r/PrivatEkonomi/comments/1d0xvxe/) and [fund depå vs ISK (2025)](https://www.reddit.com/r/PrivatEkonomi/comments/1j09g7k/)
- **The calculation depends on the exit plan and cost basis:** discussions mention the schablon method on listed fund units, the 2026 SEK 300,000 allowance, and the fact that moving an appreciated holding to ISK can trigger a taxable sale. [Buy-and-hold comparison (2026)](https://www.reddit.com/r/PrivatEkonomi/comments/1vkgqk7/) and [moving savings from AF to ISK (2026)](https://www.reddit.com/r/PrivatEkonomi/comments/1vmdeh9/)

The common thread is that people disagree because they assume different return paths, ISK tax rates, loss treatment, allowance usage, and withdrawal timing. The notebook reflects those variables in the calculations; Reddit sentiment is context only, not evidence that one account always wins.

## References and model notes

- Historical SEK prices: [`../world_fund_comparison.ipynb`](../world_fund_comparison.ipynb) and `../data/aligned_performance_sek.csv`. The original notebook converts the Xetra EUNL close from EUR into SEK with the ECB reference rate.
- iShares confirms SWDA (ISIN IE00B4L5Y983) is an accumulating share class: [iShares product page](https://www.ishares.com/uk/individual/en/products/251882/ishares-msci-world-ucits-etf-acc-fund).
- ISK calculation, tax-free allowance, and 2026 schablonintäkt: [Skatteverket legal guidance](https://www4.skatteverket.se/rattsligvagledning/edition/2026.2/2830.html) and [Skatteverket calculator](https://www.skatteverket.se/privat/skatter/vardepapper/investeringssparkontoisk/raknautschablonintakt.4.7eada0316ed67d72823b57.html).
- ISK values are measured at quarter starts: [Skatteverket guidance](https://www4.skatteverket.se/rattsligvagledning/edition/2026.14/2008.html). The November government borrowing rate is set by [Riksgälden](https://www.riksgalden.se/var-verksamhet/statslanerantan/).
- Gains on listed shares/securities and dividends are generally taxed at 30%; losses on listed securities are deductible under separate rules: [capital gains and losses](https://www.skatteverket.se/privat/etjansterochblanketter/svarpavanligafragor/vardepapper/privatvardepapperfaq/jagharsaltmarknadsnoteradeaktiernarochhurskajagdeklareraforsaljningen.5.18e1b10334ebe8bc8000119343.html) and [dividends](https://www.skatteverket.se/privat/etjansterochblanketter/svarpavanligafragor/rantorochutdelning/privatrantorfaq/vilkaskattereglergallerforranteinkomsterochutdelningar.5.18e1b10334ebe8bc8000118462.html). Skatteverket also permits the 20%-of-sale-price basis for listed fund units in its [fund rules](https://www.skatteverket.se/privat/skatter/vardepapper/omovrigavardepapper/fonder).
- Fund units in a regular account have a separate 0.4% imputed income, taxed at 30%: [Skatteverket fund rules](https://www.skatteverket.se/privat/skatter/vardepapper/omovrigavardepapper/fonder).

The tax-law rules are simplified for a transparent comparison. The model does not include personal deductions, interaction with other capital income/losses, foreign-tax-credit details, future tax-law changes, or broker costs. Its forward scenarios hold the 2026 ISK rate and allowance constant and should be read as conditional examples.